# w1-hab · Привычки и здоровье

Наблюдательный анализ связи четырёх блоков привычек — **питание, сон, стресс,
физическая активность** — с ожирением (ИМТ ≥ 30).

- Основная выборка: **NHANES 2017–2018** (измеренные ИМТ, давление, пульс;
  24-часовой дневник питания; сон; PHQ-9 как прокси стресса; активность).
- Внешние проверки: **BRFSS 2024** (стресс, активность, ИМТ) и **AI4FoodDB**
  (DASS-21, сон, IPAQ; n≈100).

Полный конвейер — в `analysis.py`; отчёт — `report/report.html`.

**Границы:** данные наблюдательные, выводы — ассоциации и гипотезы, не причинность.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import subprocess
import sys

import pandas as pd

ROOT = next(
    p
    for p in (Path.cwd(), Path.cwd() / "works" / "w1-hab")
    if (p / "data" / "PASSPORT.md").exists()
)
OUT = ROOT / "out"
RAW = ROOT / "data" / "raw"

passport = (ROOT / "data" / "PASSPORT.md").read_text(encoding="utf-8")
print(passport[:900])

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

rows = re.findall(r"\|\s*`([^`]+)`\s*\|\s*[\d,\s]+\|\s*`([0-9a-f]{64})`\s*\|", passport)
results = []
for rel, expected in rows:
    path = ROOT / rel
    actual = sha256(path) if path.exists() else "missing"
    results.append((rel, actual == expected))
    print(f"{rel:<50} {'OK' if actual == expected else 'MISMATCH'}")

assert results and all(ok for _, ok in results), "паспорт не сходится с файлами"
print(f"проверено записей: {len(results)}, все совпали")

## Запуск конвейера

`analysis.py` выполняет очистку без заполнения средним, baseline → логистическую
регрессию → permutation importance (1000 перестановок), bootstrap 1000 для OR,
традиционный анализ и внешние проверки. На этой машине занимает несколько минут.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "analysis.py")], cwd=ROOT, check=True)
print("готово:", OUT / "metrics.json", OUT / "factors.csv")

In [ ]:
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
print("n:", metrics["sample"])
print("baseline AUC:", round(metrics["model"]["baseline"]["roc_auc"], 3))
print("logit AUC:", round(metrics["model"]["logistic"]["roc_auc"], 3))
print("CV AUC:", round(metrics["model"]["cv_mean"], 3))
display(pd.DataFrame(metrics["blocks"]))

In [ ]:
factors = pd.read_csv(OUT / "factors.csv")
cols = ["factor", "block", "or_per_sd", "ci_low", "ci_high", "direction", "strength", "perm_importance"]
display(factors[cols].sort_values("perm_importance", ascending=False).head(15))

## Как читать результат

- `direction = unclear` означает, что 95% CI отношения шансов включает 1;
  такие факторы не выдаются за значимые.
- Permutation importance показывает падение ROC AUC при перемешивании
  признака/блока; отрицательные значения — шум.
- Раздел «Традиционный способ» в отчёте сравнивает доли ожирения по группам,
  простые OR и χ² без поправок.
- Все выводы — ассоциации. Причинность не устанавливается: возможны
  конфаундеры, обратная причинность и ошибка самоотчёта.